# Import Library 

In [2]:
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import os
import string as st

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Functions

In [3]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

# Import Dataset

In [23]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}
label_mapper = {
    'BT1' : 'knowledge',
    'BT2' : 'comprehension',
    'BT3' : 'application',
    'BT4' : 'analysis',
    'BT5' : 'synthesis',
    'BT6' : 'evaluation'
}


# Load dataset
df = pd.DataFrame()
for i in [1,4,3]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].replace(label_mapper)
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

print(df['label'].value_counts())

label
synthesis        158
comprehension    157
evaluation       154
analysis         149
knowledge        145
application      144
Name: count, dtype: int64


In [24]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}
label_mapper = {
    'BT1' : 'knowledge',
    'BT2' : 'comprehension',
    'BT3' : 'application',
    'BT4' : 'analysis',
    'BT5' : 'synthesis',
    'BT6' : 'evaluation'
}


# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(2) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].replace(label_mapper)
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]
print(test_df['label'].value_counts())

label
comprehension    135
application       72
evaluation        57
analysis          56
knowledge         50
synthesis         45
Name: count, dtype: int64


# EDA

## OOV test

In [25]:
def get_vocab(text_series):
    vocab = set()
    for text in text_series.dropna():
        words = text.split()
        vocab.update(words)
    return vocab

In [26]:
label_oov_dict = {}

for label in test_df['label'].unique():
    train_texts = df[df['label'] == label]['processed_question']
    test_texts = test_df[test_df['label'] == label]['processed_question']
    train_vocab = get_vocab(train_texts)
    test_vocab = get_vocab(test_texts)
    oov_words = test_vocab - train_vocab
    label_oov_dict[label] = {
        'num_oov': len(oov_words),
        'oov_words': list(oov_words)
    }

In [27]:
for label, oov_info in label_oov_dict.items():
    print(f"Label: {label}")
    print(f"  Number of OOV words: {oov_info['num_oov']}")
    print(f"  Sample OOV words: {oov_info['oov_words']}")
    print("-" * 50)

Label: synthesis
  Number of OOV words: 106
  Sample OOV words: ['security', 'aerobically', 'manage', 'mention', 'recommendation', 'annie', 'potential', 'mining', 'positive', 'carbon', 'today', 'preparation', 'protocol', 'abap', 'recovery', 'key', 'benefit', 'efficient', 'enzyme', 'impeller', 'course', 'albert', 'limitation', 'correct', 'aeration', 'acidophilus', 'sustainable', 'low', 'agitator', 'modification', 'weather', 'cultivate', 'diagram', 'rapid', 'strain', 'reason', 'butanol', 'growth', 'schema', 'where', 'call', 'applicant', 'delivery', 'computer', 'warehouse', 'discuss', 'image', 'framework', 'mall', 'cultivation', 'subroutine', 'chain', 'unable', 'desire', 'rearrange', 'evolution', 'chance', 'obtain', 'observation', 'detail', 'markov', 'agitation', 'component', 'procedure', 'video', 'skill', 'enter', 'allow', 'state', 'linear', 'digital', 'utar', 'matrix', 'k', 'interactive', 'bandwidth', 'notice', 'lactobacillus', 'communication', 'implementation', 'excrete', 'transition',